In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from reprojection import *
from utils import *
from interpolation import *
import pandas as pd

In [2]:
files = sorted(glob.glob('/home/ulyanov/data/cross/*'))
files

['/home/ulyanov/data/cross/hmi.M_45s.20241120_001500_TAI.2.magnetogram.fits',
 '/home/ulyanov/data/cross/hmi.M_720s.20241120_001200_TAI.3.magnetogram.fits',
 '/home/ulyanov/data/cross/hmi.V_45s.20241120_001500_TAI.2.Dopplergram.fits',
 '/home/ulyanov/data/cross/hmi.V_720s.20241120_001200_TAI.3.Dopplergram.fits',
 '/home/ulyanov/data/cross/solo_L2_phi-fdt-blos_20241120T001503_V202602220151_0451200501.fits.gz',
 '/home/ulyanov/data/cross/solo_L2_phi-fdt-vlos_20241120T001503_V202602220151_0451200501.fits.gz']

In [3]:
file = files[2]

with fits.open(file) as hdul:
    header = hdul[1].header
    data = hdul[1].data

data = rebin(data, 6, update_header=header)

In [4]:
view = View.from_header(header)
view_ = View.from_header(header).update(crota=180, increment=True)

In [5]:
V = (view.velocity(mu_thr=0., cbs=False) - view_.velocity(mu_thr=0., cbs=False)) / 2 / 1000
V -= np.nanmean(V)

plt.figure(figsize=(10,10))
plt.imshow(V, 'seismic', vmin=-2, vmax=2)

plt.tight_layout()

In [6]:
V_ = (data.copy() - view_.reproject(data, view)) / 2 / 1000

plt.figure(figsize=(10,10))
plt.imshow(V_, 'seismic', vmin=-2, vmax=2)

plt.tight_layout()

In [7]:
t = ~np.isnan(V)
x, y = V[t], V_[t]
t = np.where(np.abs(y - x) < 0.5)
x, y = x[t], y[t]

k = np.nanmean(x * y) / np.nanmean(x ** 2)
print(k)

plt.figure(figsize=(10,10))
plt.plot([-2,2], [-2,2], color='black', lw=0.5)
plt.scatter(V, V_, s=0.01)
plt.plot([-2,2], [-2 * k, 2 * k], '--', color='black', lw=1)


plt.xlim(-2,2)
plt.ylim(-2,2)

plt.tight_layout()

1.0012807898935943


In [8]:
V = view.velocity(mu_thr=0., cbs=True) / 1000
V -= np.nanmean(V)

V_ = data.copy() / 1000
V_[np.isnan(V)] = np.nan
V_ -= np.nanmean(V_)

In [9]:
plt.figure(figsize=(10,10))
plt.imshow(V_, 'seismic', vmin=-2, vmax=2)
plt.tight_layout()

In [10]:
plt.figure(figsize=(10,10))
plt.imshow(V_ - V * k, 'seismic', vmin=-1, vmax=1)

plt.tight_layout()

In [17]:
mu = view.mu()
U = V_ - V * k

dx = 0.1
x = np.arange(dx / 2, 1, dx)
y = []
for x_ in x:
    t = np.where(np.abs(mu - x_) < dx / 2)
    y += [np.nanmean(U[t])]
y = np.array(y)

plt.figure(figsize=(10,10))
plt.scatter(mu, U, s=0.03)
plt.plot(x, y, '--', color='black')

plt.title('Convective blueshift')
plt.xlabel(r'$\mu$')
plt.ylabel('Doppler velocity, km/s')

plt.xlim(0,1)
plt.ylim(-0.6,0.6)

plt.gca().invert_xaxis()
plt.tight_layout()

In [6]:
header

SIMPLE  =                    T / file does conform to FITS standard             
BITPIX  =                  -32 / data type of original image                    
NAXIS   =                    2 / dimension of original image                    
NAXIS1  =                  682 / length of original image axis                  
NAXIS2  =                  682 / length of original image axis                  
DATE    = '2025-01-16T01:33:47.000' / [ISO] HDU creation date                   
DATE-OBS= '2024-11-20T00:10:32.000' / [ISO] Observation date {DATE__OBS}        
TELESCOP= 'SDO/HMI '           / Telescope                                      
INSTRUME= 'HMI_COMBINED'       / For HMI: HMI_SIDE1, HMI_FRONT2, or HMI_COMBINED
WAVELNTH=                6173. / [angstrom] Wavelength                          
CAMERA  =                    3 / Camera                                         
BUNIT   = 'm/s     '           / Physical Units                                 
ORIGIN  = 'SDO/JSOC-SDP'    